# Inspect the CIO-control reconstruction

Run from the extracted package root after `pip install -e .`. This notebook reads the executed short demonstration. It does not reproduce the original manuscript results. Jupyter is optional and is not required for the command-line workflow.

In [ ]:
from pathlib import Path
import csv, json
import numpy as np
from qos_cio.config import Config
from qos_cio.plots import plot_all
root = Path.cwd()
assert (root / 'runs/demo/evaluation/episodes.csv').exists(), 'Open this notebook from the package root'


## Exact scenario and provenance

In [ ]:
config = Config.load(root / 'runs/demo/evaluation/config.yaml')
print(json.dumps(config.to_dict(), indent=2))


## Raw seed/trace outcomes

Missing values are unavailable estimands, not zeros. Always inspect delivery ratio, backlog and latency together.

In [ ]:
with (root/'runs/demo/evaluation/episodes.csv').open() as f:
    rows = list(csv.DictReader(f))
for row in rows:
    print({key: row[key] for key in ['method','training_seed','environment_seed','throughput_mbps','latency_ms','plr','backlog_packets']})


## Packet conservation

In [ ]:
for row in rows:
    assert int(row['arrived_packets']) == sum(int(row[k]) for k in ['delivered_packets','dropped_packets','backlog_packets'])
print('All evaluation rows conserve packets.')


## Paired effects

PPO minus CDQL is favorable when positive only for throughput and fairness. With two short training replicates, uncertainty is not reliable evidence of superiority.

In [ ]:
with (root/'runs/demo/evaluation/paired_differences.csv').open() as f:
    for row in csv.DictReader(f): print(row)


## Rebuild the figures from data

This writes fresh figures to a separate directory. The illustrative spatial radio background is not the time-varying simulated channel.

In [ ]:
plot_all(root/'runs/demo/evaluation', root/'runs/notebook_figures',
         [root/'runs/demo/ppo_seed_0', root/'runs/demo/ppo_seed_1'],
         root/'runs/demo/ppo_seed_0/policy.pt')
